In [ ]:
from ultralytics import YOLO

DATA = r"D:\project3_test\addcamera-7_bbox\data.yaml"

model = YOLO("yolo26n.pt")

model.train(
    data=DATA,
    epochs=100,
    imgsz=640,
    batch=16,
    workers=4,
    device=0,
    project=r"D:\project3_test\runs",
    name="addcamera-7_yolo26n",
    patience=10,
    seed=42
)

In [ ]:
from ultralytics import YOLO

DATA = r"D:\project3_test\addcamera-7_bbox\data.yaml"
WEIGHT = r"D:\project3_test\runs\addcamera-7_yolo26n\weights\best.pt"

model = YOLO(WEIGHT)

metrics = model.val(
    data=DATA,
    split="test",          # test 데이터 기준 평가
    imgsz=640,
    batch=16,
    device=0,
    project=r"D:\project3_test\runs",
    name="addcamera-7_yolo26n_test_eval"
)

print("mAP50-95:", metrics.box.map)
print("mAP50:", metrics.box.map50)
print("mAP75:", metrics.box.map75)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import shutil

WEIGHT = r"D:\project3_test\runs\addcamera-7_yolo26n\weights\best.pt"
EXPORT_DIR = r"D:\project3_test\runs\addcamera-7_yolo26n_tflite"

model = YOLO(WEIGHT)

# TFLite Float32 export
exported_path = model.export(
    format="tflite",
    imgsz=640,
    batch=1,
    half=False,   # FP16 사용 안 함
    int8=False,   # INT8 양자화 안 함
    nms=True,     # Android에서 후처리 줄이고 싶으면 True 추천
    device="cpu"  # TFLite 변환은 CPU로 해도 충분
)

print("Exported:", exported_path)

# 보기 좋게 별도 폴더로 복사
exported_path = Path(exported_path)
save_dir = Path(EXPORT_DIR)
save_dir.mkdir(parents=True, exist_ok=True)

dst = save_dir / "addcamera-7_yolo26n_float32.tflite"
shutil.copy(exported_path, dst)

print("Copied to:", dst)

In [ ]:
from ultralytics import YOLO

DATA = r"D:\project3_test\addcamera-7_bbox\data.yaml"
TFLITE_MODEL = r"D:\project3_test\runs\addcamera-7_yolo26n_tflite\addcamera-7_yolo26n_float32.tflite"

model = YOLO(TFLITE_MODEL)

metrics = model.val(
    data=DATA,
    split="test",
    imgsz=640,
    batch=1,
    device="cpu",
    project=r"D:\project3_test\runs",
    name="addcamera-7_yolo26n_tflite_float32_test_eval"
)

print("mAP50-95:", metrics.box.map)
print("mAP50:", metrics.box.map50)
print("mAP75:", metrics.box.map75)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)